In [ ]:
import os, gzip, json, re, html, unicodedata, hashlib, io

import numpy as np
import pandas as pd
import requests
import random as python_random
from tqdm import tqdm
from bs4 import BeautifulSoup
from PIL import Image

import tensorflow as tf
import tf_keras as keras
from tf_keras import layers
from tf_keras.callbacks import EarlyStopping
from transformers import TFRobertaModel, TFViTModel, RobertaTokenizer

from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

In [ ]:
def reset_random_seeds():
    tf.random.set_seed(42)
    np.random.seed(42)
    python_random.seed(42)
reset_random_seeds()

## 데이터 로드 & 전처리


In [ ]:
def load_and_preprocess(DATA_PATH):
    # # 1) gzip 리뷰 데이터 로드
    df = pd.DataFrame([json.loads(l) for l in gzip.open(DATA_PATH, "rb")])
    print("원본:", df.shape)


    # 2) 데이터 필터링
    def has_images(x):
        if isinstance(x, list):
            return len(x) > 0
        if isinstance(x, dict):
            return len(x) > 0
        return False

    mask = (
        (df["helpful_vote"] > 0) &
        (df["text"].notna()) &
        (df["text"].str.strip().str.len() > 0) &
        (df["images"].apply(has_images))
    )
    df = df[mask].reset_index(drop=True)
    print(f"필터링 후 (helpful_vote>0 & 텍스트/이미지 존재): {len(df)}개")


    # 3) 텍스트 전처리
    URL_RE  = re.compile(r"https?://\S+|www\.\S+", re.I)
    CTRL_RE = re.compile(r"[\u0000-\u001F\u007F]")
    WS_RE   = re.compile(r"\s+")

    def clean_text(text):
        if not text or not str(text).strip():
            return None
        text = str(text)
        text = html.unescape(text)
        text = BeautifulSoup(text, "html.parser").get_text()
        text = URL_RE.sub(" [URL] ", text)
        text = unicodedata.normalize("NFKC", text)
        text = CTRL_RE.sub(" ", text)
        text = WS_RE.sub(" ", text).strip().lower()
        return text if text else None

    df["clean_text"] = df["text"].apply(clean_text)
    df = df[df["clean_text"].notna() & (df["clean_text"].str.len() >= 3)].reset_index(drop=True)
    print(f"텍스트 전처리 후 (유효 텍스트 길이 >= 3): {len(df)}개")


    # 4) 첫 번째 리뷰 이미지 URL 추출
    def first_image_url(images, key="medium_image_url"):
        if isinstance(images, list) and len(images) > 0:
            img = images[0]
            if isinstance(img, dict) and img.get(key):
                return img[key]
        elif isinstance(images, dict):
            if images.get(key):
                return images[key]
            for v in images.values():
                if isinstance(v, dict) and v.get(key):
                    return v[key]
        return None

    df["image_url"] = df["images"].apply(first_image_url)
    df = df[df["image_url"].notna()].reset_index(drop=True)
    print(f"이미지 URL 추출 후: {len(df)}개")

    df["log_helpful_vote"] = np.log(df["helpful_vote"].astype(np.float32)+1)
    # 5) 필요한 컬럼만 남기기
    df = df[["clean_text", "image_url", "log_helpful_vote"]].copy()
    print(f"\n최종 데이터: {len(df)}개")
    print(f"log_helpful_vote 분포:\n{df['log_helpful_vote'].describe()}")

    return df

## 리뷰 이미지 다운로드

In [ ]:
# 6) 리뷰 이미지 다운로드 (리뷰당 첫 번째 이미지 1장)

def download_images(df, image_dir, max_workers=20):  # max_workers: 동시에 실행할 스레드 수
    os.makedirs(image_dir, exist_ok=True)

    # URL에서 이미지를 다운로드하고 PIL로 유효성 검증, 실패 시 재시도
    def download_with_retry(url, timeout=8, retries=2):
        headers = {"User-Agent": "Mozilla/5.0"}
        for _ in range(retries + 1):
            try:
                r = requests.get(url, timeout=timeout, headers=headers, stream=True)
                if r.status_code == 200 and str(r.headers.get("Content-Type", "")).startswith("image"):
                    content = r.content
                    Image.open(io.BytesIO(content)).verify()
                    return content
            except Exception:
                pass
        return None

    # 각 행마다 다운로드 작업을 처리하는 함수 (병렬로 실행됨)
    def process_row(args):
        i, row = args
        url = row["image_url"]
        m = re.search(r"\.(jpg|jpeg|png|webp|bmp|gif)(?:\?|$)", str(url), re.I)
        ext = f".{m.group(1).lower()}" if m else ".jpg"
        h = hashlib.md5((url + str(i)).encode()).hexdigest()[:6]
        fname = f"{i}_{h}{ext}"
        fpath = os.path.join(image_dir, fname)

        # 이미 다운로드된 파일이면 스킵 (재실행 시 중복 다운로드 방지)
        if os.path.exists(fpath):
            return i, fpath

        content = download_with_retry(url)
        if content is None:
            return i, None  # 실패
        else:
            with open(fpath, "wb") as f:
                f.write(content)
            return i, fpath  # 성공

    results = {}  # {원본 인덱스: 파일경로 or None} 형태로 결과 저장

    # max_workers개 스레드로 병렬 다운로드 실행
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 모든 행에 대해 process_row를 병렬로 제출
        futures = {executor.submit(process_row, (i, row)): i for i, row in df.iterrows()}
        
        # 완료되는 순서대로 결과 수집 (tqdm으로 진행률 표시)
        for future in tqdm(as_completed(futures), total=len(futures), desc="이미지 다운로드"):
            i, fpath = future.result()
            results[i] = fpath

    # 실패한 행 drop
    failed_indices = [i for i, p in results.items() if p is None]
    df = df.drop(index=failed_indices).reset_index(drop=True)

    # 원본 인덱스 순서 유지하면서 image_path 할당
    success_paths = [results[i] for i in sorted(results) if results[i] is not None]
    df["image_path"] = success_paths

    success = len(success_paths)
    fail = len(failed_indices)
    print(f"\n다운로드 완료: 성공 {success}, 실패 {fail} (실패 행 제거됨)")
    print(f"최종 데이터: {len(df)}개")
    return df

## 데이터 전처리 함수 - preprocess_image, build_dataset
- 텍스트 토큰화 → (텍스트 토큰 + 이미지경로 + 라벨)을 tf.data로 묶기 → 이미지 전처리 → 배치 → 프리페치

In [ ]:
# 샘플 하나를 모델 입력 형태로 변환
def preprocess_image(inputs, label):
    raw = tf.io.read_file(inputs["image_path"])
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, [224, 224]) / 255.0
    img = tf.transpose((img - 0.5) / 0.5, [2, 0, 1])
    return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "pixel_values": img,
    }, label

In [ ]:
def build_dataset(texts, image_paths, labels, max_length=256, batch_size=512, shuffle=False):
    # 텍스트 토큰화
    enc = tokenizer(texts, padding="max_length", max_length=max_length,
                    truncation=True, return_tensors="tf")
    
    # 입력, 딕셔너리, 라벨 쌍으로 샘플 단위로 슬라이싱
    dataset = tf.data.Dataset.from_tensor_slices((
        {"input_ids": enc["input_ids"],
         "attention_mask": enc["attention_mask"],
         "image_path": image_paths},
        labels,
    ))
    # 학습 데이터일 경우 매 epoch마다 순서를 섞음
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(labels))
    # 각 샘플에 process 전처리 적용 num_parallel_calls=AUTOTUNE로 병렬 처리
    dataset = dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    # 개별 샘플들을 배치 크기로 묶음
    dataset = dataset.batch(batch_size)
    # GPU가 현재 배치 학습하는 동안 CPU가 다음 배치를 준비
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

## 양방향 Cross-Attention
### BidirectionalCrossAttentionFusion

T_merged와 I_merged 사이에 양방향 cross-attention을 수행하는 모듈.
각 모달리티가 상대 모달리티 전체를 attention으로 흡수한 결과 (t2i, i2t)를 시퀀스 그대로 반환.
이후 외부에서 AttentionPool로 압축 → MFB로 융합됨.

In [ ]:
class BidirectionalCrossAttentionFusion(layers.Layer):
    """
    T_merged와 I_merged를 받아 양방향 cross-attention을 수행.
    각 모달리티의 cross-attended 시퀀스를 그대로 반환 (압축 / 융합 없음).

      1) t2i -> Text-to-Image Cross-Attention (Q=T, K=V=I) + FFN + Residual + LN
      2) i2t -> Image-to-Text Cross-Attention (Q=I, K=V=T) + FFN + Residual + LN

    Cross-Attention + FFN 단계별 shape:
        입력:
            text_repr  (T_merged): (B, 256, 768)
            image_repr (I_merged): (B, 197, 768)

        1단계: Text Branch
            t2i_attn = text_to_image_attn(Q=text, K=image, V=image): (B, 256, 768)
            t2i = LN(text_repr + t2i_attn): (B, 256, 768)
            t2i = LN(t2i + FFN(t2i)): (B, 256, 768)

        2단계: Image Branch
            i2t_attn = image_to_text_attn(Q=image, K=text, V=text): (B, 197, 768)
            i2t = LN(image_repr + i2t_attn): (B, 197, 768)
            i2t = LN(i2t + FFN(i2t)): (B, 197, 768)

    출력: (t2i, i2t) 튜플 — 시퀀스 차원 그대로 보존
        t2i: (B, 256, 768)  ← 텍스트 토큰 + 이미지 cross-modal 문맥
        i2t: (B, 197, 768)  ← 이미지 패치 + 텍스트 cross-modal 문맥

    설계 노트:
        본 클래스는 양방향 cross-attention + FFN까지만 담당.
        이후 단계인 시퀀스 → 단일 벡터 압축 (AttentionPool)과
        모달리티 간 융합 (MFB)은 외부 모듈로 분리되어 모델 클래스에서 호출됨
        (모듈별 책임 분리, 각 컴포넌트 교체 용이).
    """

    def __init__(self, d_model=768, num_heads=12, d_ff=3072, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        # Cross-Attention (양방향, 각 1개씩)
        self.text_to_image_attn = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout
        )
        self.image_to_text_attn = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout
        )

        # LayerNorms (Attention 뒤 + FFN 뒤, 각 브랜치에 2개씩)
        self.ln1_t = layers.LayerNormalization()
        self.ln2_t = layers.LayerNormalization()
        self.ln1_i = layers.LayerNormalization()
        self.ln2_i = layers.LayerNormalization()

        # FFN (텍스트 브랜치)
        self.ffn_t = keras.Sequential([
            layers.Dense(d_ff, activation="gelu"),
            layers.Dropout(dropout),
            layers.Dense(d_model),
        ])

        # FFN (이미지 브랜치)
        self.ffn_i = keras.Sequential([
            layers.Dense(d_ff, activation="gelu"),
            layers.Dropout(dropout),
            layers.Dense(d_model),
        ])

    def call(self, text_repr, image_repr, text_mask, training=False):
        # i2t cross-attention 시 텍스트 padding 차단을 위한 mask 생성
        # text_mask (B, 256) → (B, 1, 256) — i2t의 attention_mask 인자에 전달
        text_mask_cross = tf.expand_dims(text_mask, axis=1)

        # 1) Text branch: 텍스트가 이미지를 본 결과 (Q=text, K=V=image)
        t2i_attn = self.text_to_image_attn(
            query=text_repr, key=image_repr, value=image_repr, training=training
        )
        t2i = self.ln1_t(text_repr + t2i_attn)
        t2i = self.ln2_t(t2i + self.ffn_t(t2i, training=training))

        # 2) Image branch: 이미지가 텍스트를 본 결과 (Q=image, K=V=text, text padding mask 적용)
        i2t_attn = self.image_to_text_attn(
            query=image_repr, key=text_repr, value=text_repr,
            attention_mask=text_mask_cross, training=training
        )
        i2t = self.ln1_i(image_repr + i2t_attn)
        i2t = self.ln2_i(i2t + self.ffn_i(i2t, training=training))

        return t2i, i2t

## AttentionPool
- 시퀀스 압축용 attention pooling
- Dense(1)로 각 토큰 점수 계산 → padding mask → softmax 가중치 → 가중합
- (B, N, D) → (B, D)

각 모달리티별로 cross-attended 시퀀스를 단일 벡터로 압축. 이후 MFB의 입력으로 사용.

In [ ]:
class AttentionPool(layers.Layer):
    """
    시퀀스 압축용 기본 Attention Pooling
    (B, N, D) → (B, D)

    1) Dense(1)로 각 토큰의 중요도 스칼라 점수 계산
    2) padding mask 적용 (-1e9로 점수 매우 작게)
    3) softmax로 가중치 정규화
    4) 전체 토큰의 가중합
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.attention_fc = layers.Dense(1)

    def call(self, x, mask=None):
        # x: (B, N, D), mask: (B, N) or None

        # 1) 각 토큰의 점수 계산 (학습된 Dense(1)로 D차원 → 스칼라)
        scores = self.attention_fc(x)              # (B, N, 1)
        scores = tf.squeeze(scores, axis=-1)       # (B, N)

        # 2) padding 위치 점수를 매우 작게
        if mask is not None:
            mask_f = tf.cast(mask, scores.dtype)
            scores = scores + (1.0 - mask_f) * -1e9

        # 3) softmax로 가중치 정규화 (합=1)
        weights = tf.nn.softmax(scores, axis=-1)   # (B, N)

        # 4) 가중합 (스칼라 가중치 × D차원 토큰 벡터)
        pooled = tf.reduce_sum(
            tf.expand_dims(weights, axis=-1) * x,
            axis=1
        )                                           # (B, D)

        return pooled

## MFB (Multi-modal Factorized Bilinear Pooling)
- Yu et al., ICCV 2017 — 두 모달리티 feature를 bilinear pooling으로 융합
- 단순 concat 대비 모달리티 간 곱셈적(AND-like) 상호작용 학습
- 수식: z = SumPool(U·v_t ⊙ V·v_i, k) → power norm → L2 norm

본 모델에선 AttentionPool로 압축된 두 단일 벡터 (B, 768) × 2를 입력 받아, bilinear interaction 후 (B, mfb_dim) 반환.

**표준 구현 (Ren et al. 2024 / 원 MFB 논문 동일)**:
- mfb_dim: 최종 출력 차원 (= bilinear interaction 패턴 개수)
- k_factors: 각 패턴의 rank (factorization 정도)
- output_proj 없음 — sum pool 후 normalization만 적용

In [ ]:
class MFB(layers.Layer):
    """
    Multi-modal Factorized Bilinear Pooling (Yu et al., ICCV 2017) — 표준 구현

    두 모달리티 벡터를 bilinear pooling으로 융합.

    Bilinear weight를 W ≈ U·V^T 로 분해 (factorize)하여 파라미터 폭발 방지:
        T (B, D_T) → proj_t → (B, k_factors · mfb_dim)
        I (B, D_I) → proj_i → (B, k_factors · mfb_dim)
        hadamard product → (B, k_factors · mfb_dim)
        reshape → (B, mfb_dim, k_factors)
        sum over k_factors → (B, mfb_dim)
        power norm + L2 norm → (B, mfb_dim)

    출력: (B, mfb_dim)  ← 추가 projection 없음 (표준 MFB)

    Hyperparameters:
        mfb_dim: 최종 출력 차원 (= bilinear interaction 패턴 개수)
        k_factors: 각 패턴의 rank (factorization 정도)
    """
    def __init__(self, mfb_dim=1000, k_factors=5, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mfb_dim = mfb_dim
        self.k_factors = k_factors

        # Expand stage: T, I → k_factors × mfb_dim 차원으로 projection
        # use_bias=False — 표준 MFB 논문 / Ren et al. 2024 따름
        self.proj_t = layers.Dense(k_factors * mfb_dim, use_bias=False)
        self.proj_i = layers.Dense(k_factors * mfb_dim, use_bias=False)

        self.dropout = layers.Dropout(dropout)

    def call(self, inputs, training=False):
        # inputs: [text_vec, image_vec], 각 (B, D)
        T, I = inputs

        # 1) Expand stage — k·mfb_dim 차원 factor space로 projection
        proj_t = self.proj_t(T)   # (B, k_factors × mfb_dim)
        proj_i = self.proj_i(I)   # (B, k_factors × mfb_dim)

        # 2) Hadamard product (element-wise multiply, bilinear interaction)
        hadamard = tf.multiply(proj_t, proj_i)               # (B, k_factors × mfb_dim)
        hadamard = self.dropout(hadamard, training=training)

        # 3) Squeeze stage — Sum pool: k개 factor를 합산
        # (B, k·mfb_dim) → (B, mfb_dim, k_factors) → sum axis=-1 → (B, mfb_dim)
        sum_pooled = tf.reduce_sum(
            tf.reshape(hadamard, (-1, self.mfb_dim, self.k_factors)),
            axis=-1,
        )                                                     # (B, mfb_dim)

        # 4) Power normalization: z ← sign(z) · sqrt(|z|)
        sum_pooled = tf.sign(sum_pooled) * tf.sqrt(tf.abs(sum_pooled) + 1e-8)

        # 5) L2 normalization: z ← z / ‖z‖
        sum_pooled = tf.math.l2_normalize(sum_pooled, axis=-1)

        return sum_pooled                                     # (B, mfb_dim)

## 전체 모델


In [ ]:
class MultimodalReviewHelpfulnessModel(keras.Model):
    """
Multimodal Review Helpfulness Prediction Model (AttentionPool + MFB)

전체 shape 흐름:

1단계: 인코더 (Frozen)
    input_ids: (B, 256)
    attention_mask: (B, 256)
    pixel_values: (B, 3, 224, 224)
    
    text_hidden: 13개 × (B, 256, 768)   ← RoBERTa 각 층 출력
    image_hidden: 13개 × (B, 197, 768)  ← ViT 각 층 출력

2단계: 층 집계 (Mean Pooling, L4/L8/L12 → 1)
    T_merged = mean(text_layers,  axis=0): (B, 256, 768)
    I_merged = mean(image_layers, axis=0): (B, 197, 768)
    → 시퀀스 차원은 보존, 층 차원만 3 → 1로 압축 (3:1, 정보 손실 작음)

3단계: BidirectionalCrossAttentionFusion (1번)
    t2i: (B, 256, 768)  ← 텍스트 토큰 + 이미지 cross-modal 문맥
    i2t: (B, 197, 768)  ← 이미지 패치 + 텍스트 cross-modal 문맥

4단계: AttentionPool per modality (시퀀스 → 단일 벡터)
    t2i_pooled = AttentionPool(t2i, mask=text_mask): (B, 768)
    i2t_pooled = AttentionPool(i2t, mask=None):       (B, 768)

5단계: MFB Fusion (Bilinear Interaction) — 표준 구현
    fused = MFB([t2i_pooled, i2t_pooled]): (B, mfb_dim)
    예: mfb_dim=1000 → (B, 1000)
    → power norm + L2 norm 까지 포함 (별도 LN 불필요)

6단계: MLP Prediction Head
    Dense(mlp_hidden, relu): (B, mlp_hidden)
    Dropout(dropout)
    Dense(mlp_hidden//2, relu)
    Dropout(dropout)
    Dense(1): (B, 1)  ← 최종 helpfulness 점수
"""

    EXTRACT_LAYERS = [4, 8, 12]

    def __init__(
        self,
        roberta_name="roberta-base",
        vit_name="google/vit-base-patch16-224",
        d_model=768,
        num_heads=12,
        d_ff=3072, 
        dropout=0.1,
        mlp_hidden=256,
        mfb_dim=1000,
        k_factors=5,
        **kwargs,
    ):
        super().__init__(**kwargs)

        # 1) encoder (frozen)
        self.text_encoder = TFRobertaModel.from_pretrained(
            roberta_name, output_hidden_states=True
        )
        self.image_encoder = TFViTModel.from_pretrained(
            vit_name, output_hidden_states=True
        )
        self.text_encoder.trainable = False
        self.image_encoder.trainable = False

        # 2) cross-attention (1 module, 층 집계 후 한 번만)
        self.cross_attn = BidirectionalCrossAttentionFusion(
            d_model, num_heads, d_ff, dropout, name="cross_attn"
        )

        # 3) AttentionPool per modality
        self.text_pool = AttentionPool(name="text_pool")
        self.image_pool = AttentionPool(name="image_pool")

        # 4) MFB fusion (표준 구현 — output_proj 없음, L2 norm으로 정규화)
        self.mfb = MFB(
            mfb_dim=mfb_dim,
            k_factors=k_factors,
            dropout=dropout,
            name="mfb",
        )

        # 5) MLP head
        self.mlp = keras.Sequential([
            layers.Dense(mlp_hidden, activation="relu"),
            layers.Dropout(dropout),
            layers.Dense(mlp_hidden // 2, activation="relu"),
            layers.Dropout(dropout),
            layers.Dense(1),
        ], name="mlp_head")

    def call(self, inputs, training=False):
        # 1단계 텍스트(RoBERTa), 이미지(ViT) 인코더
        # backbone은 frozen이므로 dropout/normalization 비활성화 위해 training=False 고정
        text_out = self.text_encoder(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            training=False,
        )
        image_out = self.image_encoder(
            pixel_values=inputs["pixel_values"],
            training=False,
        )
        text_hidden = text_out.hidden_states     # 13 × (B, 256, 768)
        image_hidden = image_out.hidden_states   # 13 × (B, 197, 768)

        # 2단계 L4, L8, L12 mean-pooling으로 텍스트끼리/이미지끼리 1개로 집계
        # 시퀀스 차원은 보존, 층 차원만 평균 (3 → 1)
        text_layers = tf.stack(
            [text_hidden[idx] for idx in self.EXTRACT_LAYERS], axis=0
        )  # (3, B, 256, 768)
        image_layers = tf.stack(
            [image_hidden[idx] for idx in self.EXTRACT_LAYERS], axis=0
        )  # (3, B, 197, 768)

        T_merged = tf.reduce_mean(text_layers,  axis=0)  # (B, 256, 768)
        I_merged = tf.reduce_mean(image_layers, axis=0)  # (B, 197, 768)

        # 3단계 양방향 Cross-Attention (1번)
        # 출력: t2i (B, 256, 768), i2t (B, 197, 768) — 시퀀스 보존
        t2i, i2t = self.cross_attn(
            T_merged, I_merged, inputs["attention_mask"], training=training
        )

        # 4단계 AttentionPool per modality — 시퀀스를 단일 벡터로 압축
        # 텍스트는 padding mask 적용, 이미지는 mask=None
        t2i_pooled = self.text_pool(t2i, mask=inputs["attention_mask"])  # (B, 768)
        i2t_pooled = self.image_pool(i2t, mask=None)                     # (B, 768)

        # 5단계 MFB Fusion — bilinear interaction (표준 구현)
        # 출력 (B, mfb_dim) — 내부에 power norm + L2 norm 포함
        fused = self.mfb([t2i_pooled, i2t_pooled], training=training)    # (B, mfb_dim)

        # 6단계 MLP → helpfulness 점수
        score = self.mlp(fused, training=training)                        # (B, 1)
        return score

In [ ]:
# 모델 생성
model = MultimodalReviewHelpfulnessModel()

# 더미 입력으로 모델 빌드 & 확인 (실제 학습과 동일한 max_length=256)
dummy_inputs = {
    "input_ids": tf.random.uniform((2, 256), maxval=50265, dtype=tf.int32),
    "attention_mask": tf.ones((2, 256), dtype=tf.int32),
    "pixel_values": tf.random.normal((2, 3, 224, 224)),
}

output = model(dummy_inputs)
model.summary()
print(f"Output shape: {output.shape}")  # (2, 1)

# DATA_PATH, IMG_PATH
- 이 부분만 실험 환경에 맞게 조절

In [ ]:
DATA_PATH = "./data/Sports_and_Outdoors.jsonl.gz"
IMG_PATH = "./data/img"
df = load_and_preprocess(DATA_PATH)

In [ ]:
df = download_images(df, IMG_PATH)
df.head()

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# 데이터 준비
texts       = df["clean_text"].tolist()
image_paths = df["image_path"].tolist()
labels      = df["log_helpful_vote"].values.astype(np.float32)

# 1차 분할: train / test (80 / 20)
train_texts, test_texts, train_imgs, test_imgs, train_labels, test_labels = \
    train_test_split(texts, image_paths, labels, test_size=0.2, random_state=42)

# 2차 분할: train_inner / val (전체 기준 70 / 10 / 20)
train_texts, val_texts, train_imgs, val_imgs, train_labels, val_labels = \
    train_test_split(train_texts, train_imgs, train_labels, test_size=0.125, random_state=42)

# Dataset 생성
train_dataset = build_dataset(train_texts, train_imgs, train_labels, batch_size=512, shuffle=True)
val_dataset   = build_dataset(val_texts,   val_imgs,   val_labels,   batch_size=512)
test_dataset  = build_dataset(test_texts,  test_imgs,  test_labels,  batch_size=512)

print(f"Train: {len(train_labels)}개, Val: {len(val_labels)}개, Test: {len(test_labels)}개")

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4), 
              loss='mean_squared_error', metrics=['mean_absolute_error', 'mean_squared_error'])

early_stopping = EarlyStopping(monitor="val_loss", patience=5, verbose=1, restore_best_weights=True, mode='min')

In [ ]:
history = model.fit(train_dataset, validation_data=val_dataset, epochs=100, callbacks=[early_stopping])

In [ ]:
predicted_ratings = model.predict(test_dataset).flatten()
test_y = np.concatenate([y.numpy() for _, y in test_dataset])

## Evaluate

In [ ]:
mae = mean_absolute_error(test_y, predicted_ratings)
mse = mean_squared_error(test_y, predicted_ratings)
rmse = np.sqrt(mse)
mape = 100 * mean_absolute_percentage_error(test_y, predicted_ratings)

print(f'{mae:.4f}')
print(f'{mse:.4f}')
print(f'{rmse:.4f}')
print(f'{mape:.4f}')